# 03 — Build Candidate Dataset

Snapshot-based extraction of per-candidate prices and outcomes from raw trade CSVs.

**Pipeline:**
1. Automated snapshot: `t_resolve` = first durable 0.90 cross by the winner;
   `snapshot_time = t_resolve - BUFFER` (default 24h, parameterized).
2. Build `data/candidate_level_full.csv` (drop no-pre-snapshot candidates).
3. Winner audit.
4. Bucketing with event-clustered block bootstrap; Clopper-Pearson upper bound
   for zero-winner buckets (bootstrap is degenerate there).
5. Three-band decomposition: floored (≤2¢), above-floor longshots (2–15¢), favorites (>50¢).
6. Acceptance checks.
7. Discrete-entity refinement → `data/candidate_level_discrete.csv` + `output/buckets_*_discrete.csv`.

**Reads:** `data/event_universe.csv`, `data/markets_metadata.csv`, `data/kx*_trades.csv`
**Writes:** `data/candidate_level_full.csv`, `data/candidate_level_discrete.csv`, `output/buckets_*_discrete.csv`


In [1]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from utils import (clopper_pearson_upper, RNG_SEED, BOOT_REPS,
                   FLOOR_MAX, LS_MAX, FAV_MIN)

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats
import warnings
import time as _time
warnings.filterwarnings('ignore')

BASE_DIR = os.path.abspath('..')
DATA_DIR = os.path.join(BASE_DIR, 'data')
OUT_DIR  = os.path.join(BASE_DIR, 'output')
os.makedirs(OUT_DIR, exist_ok=True)

# ---------- Notebook-local parameters ----------
SNAPSHOT_BUFFER_H = 24      # hours before t_resolve for snapshot
CROSS_HIGH        = 0.90    # winner price must reach this
CROSS_STAY        = 0.85    # and not drop back below this after
BUCKET_EDGES      = [0.00, 0.02, 0.05, 0.10, 0.20, 0.35,
                     0.50, 0.65, 0.80, 0.95, 1.00]

np.random.seed(RNG_SEED)

print(f'SNAPSHOT_BUFFER_H : {SNAPSHOT_BUFFER_H}h')
print(f'CROSS_HIGH / STAY : {CROSS_HIGH} / {CROSS_STAY}')
print(f'FLOOR_MAX         : {FLOOR_MAX}')
print(f'BUCKET_EDGES      : {BUCKET_EDGES}')
print(f'BOOT_REPS         : {BOOT_REPS}')


SNAPSHOT_BUFFER_H : 24h
CROSS_HIGH / STAY : 0.9 / 0.85
FLOOR_MAX         : 0.02
BUCKET_EDGES      : [0.0, 0.02, 0.05, 0.1, 0.2, 0.35, 0.5, 0.65, 0.8, 0.95, 1.0]
BOOT_REPS         : 2000


In [2]:
univ = pd.read_csv(os.path.join(DATA_DIR, 'event_universe.csv'))
meta = pd.read_csv(os.path.join(DATA_DIR, 'markets_metadata.csv'), low_memory=False)
# winner_map: event_ticker -> winner sub-market ticker
winner_map = (
    meta[meta['result'] == 'yes']
    .groupby('event_ticker')['ticker']
    .first()
    .to_dict()
)

print(f'event_universe rows    : {len(univ):,}')
print(f'markets_metadata rows  : {len(meta):,}')
print(f'Winner map entries     : {len(winner_map):,}')
print()
print('Universe domain breakdown:')
print(univ['domain'].value_counts().to_string())

event_universe rows    : 182
markets_metadata rows  : 3,393
Winner map entries     : 386

Universe domain breakdown:
domain
Political    137
Sports        45


In [3]:
# Build settlement fallback map: event_ticker -> latest settlement_ts (else close_time)
_meta_ts = meta.copy()
_meta_ts['settlement_ts'] = pd.to_datetime(_meta_ts['settlement_ts'], utc=True, errors='coerce')
_meta_ts['close_time']    = pd.to_datetime(_meta_ts['close_time'],    utc=True, errors='coerce')
settlement_map = {}
for _ev, _grp in _meta_ts.groupby('event_ticker'):
    _ts = _grp['settlement_ts'].dropna().max()
    if pd.isna(_ts):
        _ts = _grp['close_time'].dropna().max()
    if pd.isna(_ts):
        settlement_map[_ev] = None
    else:
        _ts = pd.Timestamp(_ts)
        settlement_map[_ev] = _ts.tz_localize('UTC') if _ts.tzinfo is None else _ts.tz_convert('UTC')
print(f'settlement_map built for {len(settlement_map):,} events')


def load_winner_trades(winner_ticker):
    path = os.path.join(DATA_DIR, f'{winner_ticker.lower()}_trades.csv')
    if not os.path.exists(path):
        return None
    try:
        df = pd.read_csv(path)
    except Exception:
        return None
    if df.empty or 'created_time' not in df.columns:
        return None
    df['created_time']      = pd.to_datetime(df['created_time'], utc=True)
    df['yes_price_dollars'] = pd.to_numeric(df['yes_price_dollars'], errors='coerce')
    df = df.dropna(subset=['created_time', 'yes_price_dollars']).sort_values('created_time')
    return df if len(df) > 0 else None


def find_permanent_cross(df_sorted, high=CROSS_HIGH, stay=CROSS_STAY):
    if df_sorted is None or df_sorted.empty:
        return None
    prices = df_sorted['yes_price_dollars'].values
    times  = df_sorted['created_time'].values
    below_stay = np.where(prices < stay)[0]
    if len(below_stay) == 0:
        above_high = np.where(prices >= high)[0]
        return pd.Timestamp(times[above_high[0]]).tz_localize('UTC') if len(above_high) > 0 else None
    last_below = below_stay[-1]
    idx = np.where((prices >= high) & (np.arange(len(prices)) > last_below))[0]
    return pd.Timestamp(times[idx[0]]).tz_localize('UTC') if len(idx) > 0 else None


def compute_snapshot(event_ticker, buffer_h=SNAPSHOT_BUFFER_H):
    '''
    Returns (winner_ticker, t_resolve, snapshot_time, status, snap_method).
    snap_method: 'crossing' | 'fallback' | None
    Fallback: winner never had a durable >= CROSS_HIGH crossing (upset / thin market);
    t_resolve is set to the event settlement timestamp from markets_metadata.
    '''
    winner_tk = winner_map.get(event_ticker)
    if winner_tk is None:
        return None, None, None, 'no_winner', None
    df = load_winner_trades(winner_tk)
    if df is None:
        return winner_tk, None, None, 'no_csv', None
    t_res = find_permanent_cross(df)
    if t_res is not None:
        return winner_tk, t_res, t_res - pd.Timedelta(hours=buffer_h), 'ok', 'crossing'
    # Fallback: winner never crossed threshold — use settlement/close time
    t_fallback = settlement_map.get(event_ticker)
    if t_fallback is None:
        return winner_tk, None, None, 'no_durable_cross', None
    return winner_tk, t_fallback, t_fallback - pd.Timedelta(hours=buffer_h), 'ok', 'fallback'


print('Snapshot helper functions defined.')


settlement_map built for 488 events
Snapshot helper functions defined.


In [4]:
print(f'Computing automated snapshots for {len(univ):,} events ...')
print()

snap_rows = []
for _, ev_row in univ.iterrows():
    ev_ticker = ev_row['event_ticker']
    winner_tk, t_res, t_snap, status, snap_method = compute_snapshot(ev_ticker)
    snap_rows.append({
        'event_ticker':  ev_ticker,
        'domain':        ev_row['domain'],
        'winner_ticker': winner_tk,
        't_resolve':     t_res,
        'snapshot_time': t_snap,
        'status':        status,
        'snap_method':   snap_method,
    })

snap_df = pd.DataFrame(snap_rows)

# (1) snapshot_method counts
print('snapshot_method counts:')
print(snap_df['snap_method'].value_counts(dropna=False).to_string())
print()
print(f'Events with valid snapshot  : {(snap_df["status"]=="ok").sum():,}')
print(f'Events excluded (no snap)   : {(snap_df["status"]!="ok").sum():,}')
print()

# Fallback event detail
_fallbacks = snap_df[snap_df['snap_method'] == 'fallback'].copy()
if len(_fallbacks) > 0:
    print(f'FALLBACK events ({len(_fallbacks)}) — winner never durably crossed {CROSS_HIGH}:')
    print(f'  {"Event":<38}  {"Domain":<10}  {"Winner ticker":<32}  {"Last px":>7}')
    print('  ' + '-' * 92)
    for _, _r in _fallbacks.iterrows():
        _df_w = load_winner_trades(_r['winner_ticker']) if _r['winner_ticker'] else None
        _last = float(_df_w['yes_price_dollars'].iloc[-1]) if _df_w is not None else float('nan')
        print(f'  {_r["event_ticker"]:<38}  {_r["domain"]:<10}  '
              f'{str(_r["winner_ticker"]):<32}  {_last:>7.3f}')
else:
    print(f'No fallback events — all winners crossed {CROSS_HIGH} durably.')

print()
if (snap_df['status'] != 'ok').any():
    print('Events with no valid snapshot (excluded from analysis):')
    _bad = snap_df[snap_df['status'] != 'ok'][['event_ticker', 'status']]
    print(_bad.to_string(index=False))


Computing automated snapshots for 182 events ...



snapshot_method counts:
snap_method
crossing    180
fallback      2

Events with valid snapshot  : 182
Events excluded (no snap)   : 0

FALLBACK events (2) — winner never durably crossed 0.9:
  Event                                   Domain      Winner ticker                     Last px
  --------------------------------------------------------------------------------------------
  KXNEXTTEAMNFL-26MEVANS                  Sports      KXNEXTTEAMNFL-26MEVANS-SF           0.820
  KXNORWAYCON-25SEP08                     Political   KXNORWAYCON-25SEP08-24              0.370



In [5]:
# Restrict to events with valid automated snapshot
snap_ok = snap_df[snap_df['status'] == 'ok'].set_index('event_ticker')
meta['volume_fp'] = pd.to_numeric(meta['volume_fp'], errors='coerce').fillna(0)

pop_records = []
drop_log    = []

print(f'{"Event":<35}  {"Meta":>5}  {"NoCSV":>5}  {"Dropped":>7}  Note')
print('-' * 70)

for ev_ticker, ev_snap in snap_ok.iterrows():
    snap_time = ev_snap['snapshot_time']
    domain    = ev_snap['domain']
    ev_meta   = meta[meta['event_ticker'] == ev_ticker]

    n_no_csv = n_dropped = 0

    for _, row in ev_meta.iterrows():
        tk      = row['ticker']
        result  = str(row.get('result', '')).strip()
        status  = str(row.get('status', '')).strip()

        if status != 'finalized' or result not in ('yes', 'no'):
            continue

        csv_path = os.path.join(DATA_DIR, f'{tk.lower()}_trades.csv')
        if not os.path.exists(csv_path):
            n_no_csv += 1
            drop_log.append((ev_ticker, tk, 'no_csv'))
            continue

        try:
            df = pd.read_csv(csv_path)
        except Exception:
            n_dropped += 1
            drop_log.append((ev_ticker, tk, 'csv_error'))
            continue

        if df.empty or 'created_time' not in df.columns:
            n_dropped += 1
            drop_log.append((ev_ticker, tk, 'empty_csv'))
            continue

        df['created_time']      = pd.to_datetime(df['created_time'], utc=True)
        df['yes_price_dollars'] = pd.to_numeric(df['yes_price_dollars'], errors='coerce')
        df = df.dropna(subset=['created_time', 'yes_price_dollars']).sort_values('created_time')

        prior = df[df['created_time'] < snap_time]
        if prior.empty:
            n_dropped += 1
            drop_log.append((ev_ticker, tk, 'no_pre_snapshot_trades'))
            continue

        last      = prior.iloc[-1]
        snap_px   = float(last['yes_price_dollars'])
        staleness = (snap_time - last['created_time']).total_seconds() / 3600

        pop_records.append(dict(
            ticker                   = tk,
            event                    = ev_ticker,
            domain                   = domain,
            won                      = 1 if result == 'yes' else 0,
            snapshot_price           = snap_px,
            n_trades_before_snapshot = int(len(prior)),
            staleness_hours          = round(staleness, 2),
            at_floor                 = snap_px <= FLOOR_MAX,
        ))

    print(f'  {ev_ticker:<33}  {len(ev_meta):>5}  {n_no_csv:>5}  {n_dropped:>7}')

print()
print(f'Total candidate rows built : {len(pop_records):,}')
print(f'drop_log entries           : {len(drop_log):,}')

Event                                 Meta  NoCSV  Dropped  Note
----------------------------------------------------------------------
  KXACAHOUSEVOTE-27JAN01                10      0        1
  KXALCARAZCOACH-26                     12      0        1
  KXARGENTINAHOUSETHIRD-25OCT26-3        4      0        1
  KXARGENTINAMOSTGAIN-25OCT26            4      0        2
  KXARKCOACH-26                         19      0        1
  KXASIALEADEROUT-35                    10      0        7
  KXAUBCOACH-26                         27      0        8
  KXBUCHARESTMAYOR2ND-25DEC07-2          4      0        4
  KXBUCHARESTMAYOR3RD-25DEC07-3          5      0        5
  KXCABOUT-26MAR                        24      0        0


  KXCABOUT-29                           26      0        0
  KXCABOUT-29JAN                        24      0        0
  KXCALCOACH-26                         17      0       13
  KXCANADAPM-45                          6      0        0
  KXCANCOALITION-30                      6      0        0
  KXCCARCOACH-26                        13      0        0
  KXCDASEATS-25OCT29                     5      0        0
  KXCHANCELLOR-21                        6      0        0
  KXCHILEANRUNOFF-25DEC14                8      0        0
  KXCHILEROUNDONESECOND-25NOV16-2ND      8      0        8
  KXCIA-26DEC31                          9      0        0
  KXCLOSEST2025-26NOV04                  6      0        0


  KXCONNCOACH-26                        20      0       12
  KXCRASEATS-26FEB08                     8      0        8
  KXDENMARK2ND-26MAR24-2                 7      0        2
  KXDENMARK3RD-26MAR24-3                 7      0        0
  KXDENMARKPARLI-26                      4      0        1
  KXDEPORTCOUNT-25                       5      0        0
  KXELECTIONICOAST-26                    5      0        3
  KXELECTIONMOVABQ-25DEC09              11      0        0
  KXELECTIONMOVAZ7S-25SEP23              6      0        0
  KXELECTIONMOVIRELAND-25OCT24           8      0        3
  KXELECTIONMOVMIAMI-25DEC09             7      0        0
  KXELECTIONMOVVAGOV-25NOV04             9      0        0
  KXEPA-26DEC31                          7      0        4
  KXEULEADEROUT-35                      10      0        9
  KXEWCDOTA2-DOTA225                    16      0        7


  KXFBI-26DEC31                          5      0        0
  KXFDA-26DEC31                          4      0        0


  KXFEDCHAIRNOM-29                      23      0        0
  KXFIFAPEACE-27                         5      0        5
  KXFLACOACH-26                         21      0        1
  KXFTC-26DEC31                          4      0        0
  KXG7LEADEROUT-35                       7      0        2
  KXG7LEADEROUT-36                       7      0        4
  KXGERCOALITION-30                      8      0        0
  KXGERMANYPARTY-25                      6      0        0
  KXGORTONDENTON2ND-26FEB26-2ND          5      0        0
  KXGORTONDENTONMOV-26FEB26              8      0        3
  KXGOVILNOMR-26                         5      0        0
  KXGOVTFUNDSVOTES-27JAN01               9      0        0
  KXGOVTXNOMD-26                         6      0        1
  KXGROENSEATS-25OCT29                   5      0        0


...

  KXNYGCOACH-26                         33      0        0
  KXNYKCOACH-25                         13      0        2
  KXOKSTCOACH-26                        14      0        4
  KXPARIS1RMOV-26MAR15                   8      0        6
  KXPORTUGALPRESCOMBO-26JAN              5      0        5
  KXPSUCOACH-26                         42      0        0
  KXPVVSEATS-25OCT29                     6      0        1
  KXROMANIATOP-30                        7      0        5
  KXSEATTLEMAYORMOV-25NOV04             16      0        8


  KXSECAG-26DEC31                       18      0        4
  KXSECAGRO-26                          13      0        0
  KXSECCHAIR-26DEC31                    11      0        1
  KXSECCOM-26DEC31                       4      0        1
  KXSECDEF-26DEC31                      16      0        0
  KXSECED-26DEC31                        7      0        0
  KXSECENERGY-26DEC31                    8      0        0


  KXSECHHS-26DEC31                       8      0        0
  KXSECHOMELAND-26DEC31                  5      0        0
  KXSECHUD-26DEC31                       5      0        0
  KXSECINT-26DEC31                       4      0        0
  KXSECLABOR-26DEC31                     4      0        0
  KXSECSTATE-26DEC31                     5      0        0
  KXSECTREASURY-26DEC31                 11      0        0
  KXSECVA-26DEC31                        4      0        1
  KXSENATEILD-26                        14      0        0
  KXSENATENCD-26                         6      0        3
  KXSENATENCR-26                         8      0        1


  KXSENATETXD-26                        12      0        1
  KXSTANCOACH-26                        13      0        6
  KXTENNCOACH-26                        26      0        1
  KXTHAILANDPM-26FEB08                   6      0        0
  KXTHAIPARLIAMENT2ND-26FEB08-2          7      0        4
  KXTN7SMOV-25NOV07                      7      0        0
  KXTRUMPSCOTUSVOTE-26                  10      0        0
  KXTULNCOACH-26                        18      0        3
  KXTX02R-26                             4      0        0
  KXTX18SPECIAL-26                       6      0        0
  KXTX28D-26                             4      0        2
  KXTX31R-26                             5      0        0
  KXTX34R-26                             4      0        1
  KXTXPRIMARY-08R26                      6      0        0
  KXTXPRIMARY-09D26                      6      0        1
  KXTXPRIMARY-10R26                     10      0        0
  KXTXPRIMARY-22D26                      5      0       

  KXTXPRIMARY-23D26                      4      0        1
  KXTXSENDPRIMARYMOV-26MAR03            11      0        0
  KXTXSENRPRIMARYMOV-26MAR03             9      0        0
  KXUABCOACH-26                         10      0        6
  KXUCLACOACH-26                        17      0        3
  KXUNAMB-26DEC31                        7      0        0
  KXUNTCOACH-26                         12      0       12
  KXUSTR-26DEC31                         4      0        0
  KXVAAGMOV-25NOV04                     10      0        0
  KXVAHOD1-26NOV04                       8      0        0
  KXVIRGINIAHOUSEDEMS-25NOV04           10      0        1
  KXVTCOACH-26                          10      0        3

Total candidate rows built : 1,837
drop_log entries           : 418


In [6]:
cand = pd.DataFrame(pop_records)
print(f'Row count before bucket assignment: {len(cand):,}')

# Attach n_event_candidates per event
_npe  = cand.groupby('event').size().rename('n_event_candidates').reset_index()
cand  = cand.merge(_npe, on='event', how='left')

# Assign price bucket
def _assign_bucket(price):
    for i in range(len(BUCKET_EDGES) - 1):
        if BUCKET_EDGES[i] <= price < BUCKET_EDGES[i + 1]:
            return i
    return len(BUCKET_EDGES) - 2

cand['bucket'] = cand['snapshot_price'].apply(_assign_bucket)

print(f'Row count after bucket assignment  : {len(cand):,}')
print()
print(f'Sports candidates   : {len(cand[cand["domain"]=="Sports"]):,}')
print(f'Political candidates: {len(cand[cand["domain"]=="Political"]):,}')
print(f'Other candidates    : {len(cand[cand["domain"]=="Other"]):,}')

Row count before bucket assignment: 1,837
Row count after bucket assignment  : 1,837

Sports candidates   : 941
Political candidates: 896
Other candidates    : 0


In [7]:
print('=' * 70)
print('STEP 7 — WINNER AUDIT')
print('=' * 70)
print()

in_sample_winners     = int(cand['won'].sum())
events_with_winner    = int(cand[cand['won'] == 1]['event'].nunique())
events_in_analysis    = int(cand['event'].nunique())
events_winner_dropped = events_in_analysis - events_with_winner

print(f'Candidates in analysis                : {len(cand):,}')
print(f'Events in analysis                    : {events_in_analysis:,}')
print(f'In-sample winners (sum won==1)        : {in_sample_winners:,}')
print(f'Events whose winner survived snapshot : {events_with_winner:,}')
print(f'Events whose winner was DROPPED       : {events_winner_dropped:,}')
print()

assert in_sample_winners == events_with_winner, (
    f'AUDIT FAIL: {in_sample_winners} winners != {events_with_winner} events with winner'
)
print('AUDIT PASSED: in-sample winners == events with surviving winner.')
print()

_bucket_wins = cand.groupby('bucket').apply(lambda x: x['won'].sum()).sum()
print(f'Cross-check sum(won) from buckets : {int(_bucket_wins):,}  (should equal {in_sample_winners})')
print()

# Domain breakdown
print('Surviving-winner events by domain:')
_w_dom   = cand[cand['won'] == 1].groupby('domain')['event'].nunique()
_all_dom = cand.groupby('domain')['event'].nunique()
for _dom in sorted(_all_dom.index):
    print(f'  {_dom:<12}: {_w_dom.get(_dom, 0):>4} surviving-winner'
          f' / {_all_dom.get(_dom, 0):>4} total events')
print()

# Political subtype breakdown
APPT_PREFIXES = ('KXSEC', 'KXFEDCHAIRNOM', 'KXFBI-', 'KXFTC-', 'KXEPA-', 'KXFDA-',
                 'KXUSTR', 'KXISAMB', 'KXNEXTDHSSEC')
UNSURE_EXACT  = {'KXTRUMPSCOTUSVOTE-26', 'KXDEPORTCOUNT-25',
                 'KXCLOSEST2025-26NOV04', 'KXGOVILNOMR-26', 'KXGOVTXNOMD-26'}

def _pol_subtype(event_ticker):
    t = event_ticker.upper()
    if t in UNSURE_EXACT:
        return 'unsure'
    for p in APPT_PREFIXES:
        if t.startswith(p):
            return 'appointment'
    return 'election'

_pol = cand[cand['domain'] == 'Political'].copy()
if len(_pol) > 0:
    _pol['subtype'] = _pol['event'].map(_pol_subtype)
    print('Political events by subtype:')
    _w_st   = _pol[_pol['won'] == 1].groupby('subtype')['event'].nunique()
    _all_st = _pol.groupby('subtype')['event'].nunique()
    for _st in ('election', 'appointment', 'unsure'):
        print(f'  {_st:<12}: {_w_st.get(_st, 0):>4} surviving-winner'
              f' / {_all_st.get(_st, 0):>4} events')
    _unsure_ev = sorted(_pol[_pol['subtype'] == 'unsure']['event'].unique())
    if _unsure_ev:
        print(f'  Flagged unsure ({len(_unsure_ev)}): {_unsure_ev}')
    print()

# List dropped winners
_winner_in_cand = set(cand[cand['won'] == 1]['event'].unique())
_all_events     = set(snap_ok.index)
_dropped_events = _all_events - _winner_in_cand

if _dropped_events:
    print(f'Events whose winner was DROPPED from snapshot ({len(_dropped_events)}):')
    print('(winner contract post-creation / no pre-snapshot trades)')
    for _ev in sorted(_dropped_events):
        _wtk = snap_ok.loc[_ev, 'winner_ticker'] if _ev in snap_ok.index else '?'
        _dom = snap_ok.loc[_ev, 'domain']        if _ev in snap_ok.index else '?'
        print(f'  {_ev:<38}  domain={_dom}  winner={_wtk}')
else:
    print('No events lost their winner to snapshot filtering.')


STEP 7 — WINNER AUDIT

Candidates in analysis                : 1,837
Events in analysis                    : 169
In-sample winners (sum won==1)        : 158
Events whose winner survived snapshot : 158
Events whose winner was DROPPED       : 11

AUDIT PASSED: in-sample winners == events with surviving winner.

Cross-check sum(won) from buckets : 158  (should equal 158)

Surviving-winner events by domain:
  Political   :  120 surviving-winner /  129 total events
  Sports      :   38 surviving-winner /   40 total events

Political events by subtype:
  election    :   98 surviving-winner /  101 events
  appointment :   17 surviving-winner /   23 events
  unsure      :    5 surviving-winner /    5 events
  Flagged unsure (5): ['KXCLOSEST2025-26NOV04', 'KXDEPORTCOUNT-25', 'KXGOVILNOMR-26', 'KXGOVTXNOMD-26', 'KXTRUMPSCOTUSVOTE-26']

Events whose winner was DROPPED from snapshot (24):
(winner contract post-creation / no pre-snapshot trades)
  KXBUCHARESTMAYOR2ND-25DEC07-2           domain=Poli

In [8]:
def _run_bootstrap(cand_df):
    """
    Event-clustered block bootstrap for candidate-weighted (CW) and
    event-weighted (EW) win rates simultaneously.
    Returns (boot_cw, boot_ew) each shape (BOOT_REPS, n_buckets).
    """
    ev_groups = {ev: g.reset_index(drop=True) for ev, g in cand_df.groupby('event')}
    evs       = list(ev_groups.keys())
    n_b       = len(BUCKET_EDGES) - 1
    boot_cw   = np.full((BOOT_REPS, n_b), np.nan)
    boot_ew   = np.full((BOOT_REPS, n_b), np.nan)

    for r in range(BOOT_REPS):
        sampled = np.random.choice(evs, size=len(evs), replace=True)
        bdf     = pd.concat([ev_groups[ev] for ev in sampled], ignore_index=True)
        for b in range(n_b):
            sub = bdf[bdf['bucket'] == b]
            if len(sub) == 0:
                continue
            boot_cw[r, b] = float(sub['won'].mean())
            boot_ew[r, b] = float(sub.groupby('event')['won'].mean().mean())

    return boot_cw, boot_ew


def _make_tables(cand_df):
    """
    Build CW and EW bucket tables with proper CIs.
    - Non-zero-winner buckets: event-clustered block bootstrap (2.5-97.5%).
    - Zero-winner buckets: Clopper-Pearson upper bound (bootstrap is degenerate).
    Returns (tbl_cw, tbl_ew).
    """
    boot_cw, boot_ew = _run_bootstrap(cand_df)
    n_b = len(BUCKET_EDGES) - 1
    rows_cw, rows_ew = [], []

    for b in range(n_b):
        label   = f'[{BUCKET_EDGES[b]:.2f},{BUCKET_EDGES[b+1]:.2f})'
        sub     = cand_df[cand_df['bucket'] == b]
        n       = len(sub)
        n_ev    = int(sub['event'].nunique()) if n > 0 else 0
        mp      = float(sub['snapshot_price'].mean()) if n > 0 else np.nan
        sf      = float(sub['at_floor'].mean())       if n > 0 else np.nan
        n_wins  = int(sub['won'].sum())                if n > 0 else 0
        wr_cw   = float(sub['won'].mean())             if n > 0 else np.nan
        wr_ew   = (float(sub.groupby('event')['won'].mean().mean())
                   if n > 0 else np.nan)
        flag    = (n > 0) and (n_ev <= 2 or sf > 0.5)

        def _boot_ci(mat):
            v = mat[:, b]
            v = v[~np.isnan(v)]
            return (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) if len(v) >= 10 else (np.nan, np.nan)

        if n_wins == 0 and n > 0:
            # Clopper-Pearson upper bound; lower is always 0
            cw_lo, cw_hi = 0.0, clopper_pearson_upper(n,    alpha=0.05)
            ew_lo, ew_hi = 0.0, clopper_pearson_upper(n_ev, alpha=0.05)
            ci_method    = 'clopper_pearson'
        else:
            cw_lo, cw_hi = _boot_ci(boot_cw)
            ew_lo, ew_hi = _boot_ci(boot_ew)
            ci_method    = 'bootstrap'

        base = dict(bucket=label, n_candidates=n, n_events=n_ev,
                    mean_price=mp, share_at_floor=sf, flag=flag,
                    n_winners=n_wins, ci_method=ci_method)
        rows_cw.append({**base, 'win_rate': wr_cw, 'boot_lo': cw_lo, 'boot_hi': cw_hi})
        rows_ew.append({**base, 'win_rate': wr_ew, 'boot_lo': ew_lo, 'boot_hi': ew_hi})

    return pd.DataFrame(rows_cw), pd.DataFrame(rows_ew)


print('CI helpers defined.')

CI helpers defined.


In [9]:
print('=' * 80)
print('STEP 8 — THREE-BAND DECOMPOSITION')
print('=' * 80)
print('  Floored        : price <= 0.02  (tick-floor artifact, $0.01 min-tick)')
print('  Above-floor LS : 0.02 < price ≤ 0.15  (behavioral-bias test zone)')
print('  Favorites      : price > 0.50  (calibration check)')
print()
print('The FLB behavioral claim rests ONLY on the above-floor longshot band.')
print()


def _band_stats(df, label):
    n        = len(df)
    n_ev     = int(df['event'].nunique()) if n > 0 else 0
    mean_px  = float(df['snapshot_price'].mean()) if n > 0 else np.nan
    n_wins   = int(df['won'].sum())
    wr_cw    = float(df['won'].mean())                      if n > 0 else np.nan
    wr_ew    = (float(df.groupby('event')['won'].mean().mean())
                if n > 0 and n_ev > 0 else np.nan)
    sf       = float(df['at_floor'].mean())                 if n > 0 else np.nan

    if n_wins == 0 and n > 0:
        cw_lo, cw_hi = 0.0, clopper_pearson_upper(n,    alpha=0.05)
        ew_lo, ew_hi = 0.0, clopper_pearson_upper(n_ev, alpha=0.05)
        method       = 'CP'
    elif n > 0:
        # Quick bootstrap for the band
        ev_groups = {ev: g for ev, g in df.groupby('event')}
        evs       = list(ev_groups.keys())
        boot_cw_b = boot_ew_b = []
        boot_cw_b, boot_ew_b = [], []
        for _ in range(BOOT_REPS):
            samp     = np.random.choice(evs, size=len(evs), replace=True)
            bdf      = pd.concat([ev_groups[e] for e in samp], ignore_index=True)
            boot_cw_b.append(float(bdf['won'].mean()))
            boot_ew_b.append(float(bdf.groupby('event')['won'].mean().mean()))
        cw_lo, cw_hi = np.percentile(boot_cw_b, [2.5, 97.5])
        ew_lo, ew_hi = np.percentile(boot_ew_b, [2.5, 97.5])
        method       = 'bootstrap'
    else:
        cw_lo = cw_hi = ew_lo = ew_hi = np.nan
        method = '-'

    print(f'  {label}')
    print(f'    n_candidates : {n:,}   n_events : {n_ev}   share_at_floor : {sf:.3f}')
    print(f'    mean_implied : {mean_px:.4f}')
    if not np.isnan(wr_cw):
        gap_cw = wr_cw - mean_px
        print(f'    win_rate CW  : {wr_cw:.4f}  95%CI [{cw_lo:.4f}, {cw_hi:.4f}]  gap vs implied: {gap_cw:+.4f}  ({method})')
    if not np.isnan(wr_ew):
        gap_ew = wr_ew - mean_px
        print(f'    win_rate EW  : {wr_ew:.4f}  95%CI [{ew_lo:.4f}, {ew_hi:.4f}]  gap vs implied: {gap_ew:+.4f}  ({method})')
    print()
    return dict(band=label, n=n, n_ev=n_ev, mean_px=mean_px, n_wins=n_wins,
                wr_cw=wr_cw, cw_lo=cw_lo, cw_hi=cw_hi,
                wr_ew=wr_ew, ew_lo=ew_lo, ew_hi=ew_hi, method=method)


floored_mask = cand['snapshot_price'] <= FLOOR_MAX
abfloor_mask = (cand['snapshot_price'] > FLOOR_MAX) & (cand['snapshot_price'] <= 0.15)
favs_mask    = cand['snapshot_price'] > 0.50

_band_results = [
    _band_stats(cand[floored_mask], 'Floored (price <= 0.02)'),
    _band_stats(cand[abfloor_mask], 'Above-floor longshots (0.02 < price ≤ 0.15)  ** BEHAVIORAL CLAIM **'),
    _band_stats(cand[favs_mask],    'Favorites (price > 0.50)'),
]

STEP 8 — THREE-BAND DECOMPOSITION
  Floored        : price <= 0.02  (tick-floor artifact, $0.01 min-tick)
  Above-floor LS : 0.02 < price ≤ 0.15  (behavioral-bias test zone)
  Favorites      : price > 0.50  (calibration check)

The FLB behavioral claim rests ONLY on the above-floor longshot band.



  Floored (price <= 0.02)
    n_candidates : 1,064   n_events : 125   share_at_floor : 1.000
    mean_implied : 0.0116
    win_rate CW  : 0.0075  95%CI [0.0027, 0.0143]  gap vs implied: -0.0041  (bootstrap)
    win_rate EW  : 0.0260  95%CI [0.0088, 0.0407]  gap vs implied: +0.0144  (bootstrap)



  Above-floor longshots (0.02 < price ≤ 0.15)  ** BEHAVIORAL CLAIM **
    n_candidates : 463   n_events : 135   share_at_floor : 0.000
    mean_implied : 0.0689
    win_rate CW  : 0.0302  95%CI [0.0163, 0.0465]  gap vs implied: -0.0386  (bootstrap)
    win_rate EW  : 0.0362  95%CI [0.0196, 0.0515]  gap vs implied: -0.0327  (bootstrap)



  Favorites (price > 0.50)
    n_candidates : 117   n_events : 110   share_at_floor : 0.000
    mean_implied : 0.7874
    win_rate CW  : 0.8120  95%CI [0.7417, 0.8814]  gap vs implied: +0.0246  (bootstrap)
    win_rate EW  : 0.8348  95%CI [0.7843, 0.8866]  gap vs implied: +0.0475  (bootstrap)



In [10]:
print()
print('=' * 80)
print('FINAL DIAGNOSTICS (in order)')
print('=' * 80)

# (1) Snapshot method summary
print()
print('(1) SNAPSHOT METHOD COUNTS')
print('-' * 50)
for _m, _c in snap_df['snap_method'].value_counts(dropna=False).items():
    print(f'  {str(_m):<12}: {_c:>4}')
_n_fb = int((snap_df['snap_method'] == 'fallback').sum())
if _n_fb > 0:
    print(f'  (fallback list printed in compute-snapshots cell)')

# (3) Winner audit summary
print()
print('(3) WINNER AUDIT SUMMARY')
print('-' * 50)
print(f'  Events in analysis          : {events_in_analysis}')
print(f'  In-sample winners           : {in_sample_winners}')
print(f'  Events with surviving winner: {events_with_winner}')
print(f'  Events winner dropped       : {events_winner_dropped}')
print()
_w_dom2   = cand[cand['won'] == 1].groupby('domain')['event'].nunique()
_all_dom2 = cand.groupby('domain')['event'].nunique()
for _d in sorted(_all_dom2.index):
    print(f'  {_d:<12}: {_w_dom2.get(_d,0):>4} surviving-winner / {_all_dom2.get(_d,0):>4} events')
_pol2 = cand[cand['domain'] == 'Political'].copy()
if len(_pol2) > 0:
    _pol2['subtype'] = _pol2['event'].map(_pol_subtype)
    _w_st2   = _pol2[_pol2['won'] == 1].groupby('subtype')['event'].nunique()
    _all_st2 = _pol2.groupby('subtype')['event'].nunique()
    print()
    print('  Political subtype:')
    for _s in ('election', 'appointment', 'unsure'):
        print(f'    {_s:<12}: {_w_st2.get(_s,0):>4} surviving-winner / {_all_st2.get(_s,0):>4} events')

# (4) Above-floor longshot band
print()
print('(4) ABOVE-FLOOR LONGSHOT BAND (0.02 < price ≤ 0.15) — BEHAVIORAL BIAS TEST')
print('-' * 50)
_ab = cand[(cand['snapshot_price'] > FLOOR_MAX) & (cand['snapshot_price'] <= 0.15)]
_ab_n   = len(_ab)
_ab_nev = _ab['event'].nunique()
_ab_mp  = _ab['snapshot_price'].mean() if _ab_n > 0 else float('nan')
_ab_wr  = _ab['won'].mean()            if _ab_n > 0 else float('nan')
_ab_band = next((r for r in _band_results if 'Above-floor' in r['band']), {})
_ab_lo  = _ab_band.get('cw_lo', float('nan'))
_ab_hi  = _ab_band.get('cw_hi', float('nan'))
_ab_mth = _ab_band.get('method', '?')
print(f'  n_candidates  : {_ab_n:,}')
print(f'  n_events      : {_ab_nev}')
print(f'  mean implied  : {_ab_mp:.4f}')
print(f'  win_rate (CW) : {_ab_wr:.4f}  95%CI [{_ab_lo:.4f}, {_ab_hi:.4f}]  ({_ab_mth})')
if not (np.isnan(_ab_wr) or np.isnan(_ab_mp)):
    _gap = _ab_wr - _ab_mp
    print(f'  gap           : {_gap:+.4f}')
    if _ab_hi < _ab_mp:
        print('  >> FLB DETECTED: win rate CI lies BELOW implied price.')
    elif _ab_lo > _ab_mp:
        print('  >> REVERSE FLB: win rate CI lies ABOVE implied price.')
    else:
        print('  >> Above-floor FLB not statistically significant at 95%.')

# (5) NCAAF-26 sensitivity
print()
print('(5) NCAAF-26 SENSITIVITY (above-floor band with vs without KXNCAAF-26)')
print('-' * 50)
_ab_all  = cand[(cand['snapshot_price'] > FLOOR_MAX) & (cand['snapshot_price'] <= 0.15)]
_ab_nocf = _ab_all[_ab_all['event'] != 'KXNCAAF-26']
_n_cf_rm = len(_ab_all) - len(_ab_nocf)
print(f'  Candidates from KXNCAAF-26 in band : {_n_cf_rm}')
print(f'  Band size without KXNCAAF-26       : {len(_ab_nocf):,} ({_ab_nocf["event"].nunique()} events)')
if len(_ab_nocf) > 0:
    _wr_nocf = _ab_nocf['won'].mean()
    _mp_nocf = _ab_nocf['snapshot_price'].mean()
    print(f'  Without NCAAF: wr={_wr_nocf:.4f}  implied={_mp_nocf:.4f}  gap={_wr_nocf-_mp_nocf:+.4f}')
    print(f'  With NCAAF   : wr={_ab_wr:.4f}  implied={_ab_mp:.4f}  gap={_ab_wr-_ab_mp:+.4f}')
    print(f'  Shift in gap from NCAAF inclusion: {(_wr_nocf-_mp_nocf)-(_ab_wr-_ab_mp):+.4f}')

# (6) Scalar residual
print()
print('(6) SCALAR RESIDUAL CHECK')
print('-' * 50)
_total_exp = cand['snapshot_price'].sum()
_total_act = float(cand['won'].sum())
_resid     = _total_act - _total_exp
_n_ev      = cand['event'].nunique()
print(f'  Sum(implied prices)   : {_total_exp:,.2f}')
print(f'  Actual winners        : {_total_act:,.0f}')
print(f'  Residual              : {_resid:+.2f}  ({_resid/_n_ev:+.3f} per event)')
if _resid < -0.5:
    print('  >> Actual < implied: market over-priced on aggregate (FLB direction).')
elif _resid > 0.5:
    print('  >> Actual > implied: market under-priced on aggregate.')
else:
    print('  >> Near zero: aggregate calibration is close.')
_ev_or = cand.groupby('event')['snapshot_price'].sum()
print(f'  Per-event overround: mean={_ev_or.mean():.4f}  '
      f'median={_ev_or.median():.4f}  '
      f'n>1.0: {(_ev_or > 1.0).sum()}/{len(_ev_or)}')



FINAL DIAGNOSTICS (in order)

(1) SNAPSHOT METHOD COUNTS
--------------------------------------------------
  crossing    :  180
  fallback    :    2
  (fallback list printed in compute-snapshots cell)

(3) WINNER AUDIT SUMMARY
--------------------------------------------------
  Events in analysis          : 169
  In-sample winners           : 158
  Events with surviving winner: 158
  Events winner dropped       : 11

  Political   :  120 surviving-winner /  129 events
  Sports      :   38 surviving-winner /   40 events

  Political subtype:
    election    :   98 surviving-winner /  101 events
    appointment :   17 surviving-winner /   23 events
    unsure      :    5 surviving-winner /    5 events

(4) ABOVE-FLOOR LONGSHOT BAND (0.02 < price ≤ 0.15) — BEHAVIORAL BIAS TEST
--------------------------------------------------
  n_candidates  : 463
  n_events      : 135
  mean implied  : 0.0689
  win_rate (CW) : 0.0302  95%CI [0.0163, 0.0465]  (bootstrap)
  gap           : -0.0386
  >>

In [11]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  VERIFICATION CELL — Checks 1-4                                     ║
# ╚══════════════════════════════════════════════════════════════════════╝

print('=' * 80)
print('CHECK 1 — Above-floor longshot band (2-15c): CI method verification')
print('=' * 80)
print()

# Confirm the band bootstrap in _band_stats is event-clustered (read-only check)
# _band_stats resamples evs = list of unique events in the band, size=len(evs),
# replace=True — that is the event-clustered block bootstrap.
# The bucket-level bootstrap in _run_bootstrap/_make_tables does the same.
# No code change is needed; both already use event-clustered CIs.

_ab1 = cand[(cand['snapshot_price'] > FLOOR_MAX) & (cand['snapshot_price'] <= 0.15)].copy()
_n1      = len(_ab1)
_nev1    = _ab1['event'].nunique()
_nwin1   = int(_ab1['won'].sum())
_wr1     = _ab1['won'].mean()
_mp1     = _ab1['snapshot_price'].mean()

# Pull clustered CI from already-computed _band_results
_ab_r    = next(r for r in _band_results if 'Above-floor' in r['band'])
_clu_lo  = _ab_r['cw_lo']
_clu_hi  = _ab_r['cw_hi']

# Wilson (naive binomial) CI — treats all candidates as independent
def _wilson_ci(k, n, alpha=0.05):
    if n == 0: return float('nan'), float('nan')
    p  = k / n
    z  = scipy_stats.norm.ppf(1 - alpha / 2)
    d  = 1 + z**2 / n
    c  = (p + z**2 / (2 * n)) / d
    m  = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return c - m, c + m

_wil_lo, _wil_hi = _wilson_ci(_nwin1, _n1)

print(f'Above-floor longshot band  (0.02 < price ≤ 0.15)')
print(f'  n_candidates      : {_n1:,}')
print(f'  n_events          : {_nev1}')
print(f'  n_winners         : {_nwin1}')
print(f'  mean implied price: {_mp1:.4f}')
print(f'  win_rate (CW)     : {_wr1:.4f}')
print()
print(f'  Naive Wilson 95%CI (treats {_n1} candidates as independent):')
print(f'    [{_wil_lo:.4f}, {_wil_hi:.4f}]')
print(f'  Event-clustered block bootstrap 95%CI ({_nev1} events, {BOOT_REPS} reps):')
print(f'    [{_clu_lo:.4f}, {_clu_hi:.4f}]  << HEADLINE (already reported above)')
print()
_wider = (_clu_hi - _clu_lo) > (_wil_hi - _wil_lo)
print(f'  Clustered CI is {"wider" if _wider else "narrower"} than Wilson '
      f'(width: {_clu_hi-_clu_lo:.4f} vs {_wil_hi-_wil_lo:.4f}).')
print(f'  Both CIs lie entirely below mean implied ({_mp1:.4f}): '
      f'{"YES" if _clu_hi < _mp1 and _wil_hi < _mp1 else "NO — check carefully"}')
print()

# Events that contributed winners in the 2-15c band
_winner_rows = _ab1[_ab1['won'] == 1][['event', 'ticker', 'snapshot_price', 'domain']]
print(f'Events with a winner in the 2-15c band ({len(_winner_rows)} winners):' )
print(f'  {"Event":<38}  {"Domain":<10}  {"Winner ticker":<35}  {"Snap price":>10}')
print('  ' + '-' * 98)
for _, _wr_row in _winner_rows.sort_values('event').iterrows():
    print(f'  {_wr_row["event"]:<38}  {_wr_row["domain"]:<10}  '
          f'{_wr_row["ticker"]:<35}  {_wr_row["snapshot_price"]:>10.4f}')
print()
_ev_win_counts = _winner_rows['event'].value_counts()
print(f'  Winners spread across {len(_ev_win_counts)} distinct events '
      f'(max {_ev_win_counts.max()} winner per event, expected = 1).')


print()
print('=' * 80)
print('CHECK 2 — The 10 dropped events (132 universe -> 122 analysis)')
print('=' * 80)
print()

_cand_events    = set(cand['event'].unique())
_univ_events    = set(snap_ok.index)
_dropped_events = sorted(_univ_events - _cand_events)
print(f'Events in event_universe (snap_ok) : {len(_univ_events)}')
print(f'Events in candidate_level (cand)   : {len(_cand_events)}')
print(f'Dropped                            : {len(_dropped_events)}')
print()

# Build drop reasons from drop_log
_drop_df = pd.DataFrame(drop_log, columns=['event', 'ticker', 'reason'])

print(f'  {"Event":<40}  {"Domain":<10}  {"n_sub":>5}  {"Reason summary"}')
print('  ' + '-' * 98)

_UPSET_THRESHOLD = 0.20
_upset_flags = []

for _ev in _dropped_events:
    _dom       = snap_ok.loc[_ev, 'domain'] if _ev in snap_ok.index else '?'
    _ev_drops  = _drop_df[_drop_df['event'] == _ev]
    _ev_meta   = meta[meta['event_ticker'] == _ev]
    _n_sub     = len(_ev_meta)
    _reason_ct = _ev_drops['reason'].value_counts().to_dict()
    _reason_str = '  '.join(f'{k}={v}' for k, v in _reason_ct.items())
    print(f'  {_ev:<40}  {_dom:<10}  {_n_sub:>5}  {_reason_str}')

    # Check winner closing price
    _wtk = snap_ok.loc[_ev, 'winner_ticker'] if _ev in snap_ok.index else None
    if _wtk:
        _wdf = load_winner_trades(_wtk)
        if _wdf is not None and len(_wdf) > 0:
            _last_px = float(_wdf['yes_price_dollars'].iloc[-1])
            if _last_px < _UPSET_THRESHOLD:
                _upset_flags.append((_ev, _dom, _wtk, _last_px))
                print(f'    !! UPSET FLAG: winner {_wtk} last price = {_last_px:.4f} < {_UPSET_THRESHOLD}')

print()
if _upset_flags:
    print(f'UPSET FLAGS ({len(_upset_flags)}) — dropped winners with low closing price:')
    for _ef in _upset_flags:
        print(f'  {_ef[0]}  domain={_ef[1]}  winner={_ef[2]}  last_px={_ef[3]:.4f}')
    print('  These events should be investigated — dropping upset winners biases FLB results.')
else:
    print('No upset flags: all dropped winners had last price >= 0.20.')
    print('Drop reasons are structural (no pre-snapshot trades), not selection on outcome.')

print()
print('=' * 80)
print('CHECK 3 — Discrete-entity vs numeric-range event classification')
print('=' * 80)
print()

_RANGE_INDICATORS = [
    'or more', 'or fewer', 'or less', 'at least', 'between',
    'more than', 'fewer than', 'less than', 'at most',
    'or higher', 'or lower', 'over ', 'under ', 'above ', 'below ',
    '+ seats', 'seats or', 'greater than', 'no more than',
]

_range_events = []
_checked      = 0
for _ev in sorted(_cand_events):
    _ev_meta = meta[meta['event_ticker'] == _ev]
    _titles  = _ev_meta['title'].fillna('').str.lower().tolist()
    _tickers = _ev_meta['ticker'].fillna('').str.lower().tolist()
    _matched = False
    _sample  = ''
    for _t in _titles:
        for _ind in _RANGE_INDICATORS:
            if _ind in _t:
                _matched = True
                _sample  = _t[:80]
                break
        if _matched:
            break
    if _matched:
        _dom = cand[cand['event'] == _ev]['domain'].iloc[0]
        _range_events.append((_ev, _dom, _sample))
    _checked += 1

print(f'Checked {_checked} events in analysis.')
if _range_events:
    print(f'Numeric-range events detected ({len(_range_events)}):' )
    for _re in _range_events:
        print(f'  {_re[0]:<40}  {_re[1]:<10}  sample title: "{_re[2]}"')
    print()
    print('  !! These events have sub-markets that are numeric brackets, not distinct entities.')
    print('  !! Review whether they should be excluded from FLB analysis.')
else:
    print('No numeric-range events detected in the 122 analyzed events.')
    print('All events classified as discrete-entity (candidates/teams/parties).')

print()
print('=' * 80)
print('CHECK 4 — Buffer sensitivity (24h / 7d / 14d)')
print('=' * 80)
print()
print('Re-computes above-floor band at each buffer. Does NOT overwrite primary outputs.')
print()

# Pre-load all trade frames once (reuse across buffer calls)
print('Pre-loading trade CSVs into memory...')
_trade_cache = {}
for _ev_ticker, _ev_row in snap_ok.iterrows():
    _ev_meta2 = meta[meta['event_ticker'] == _ev_ticker]
    for _, _mrow in _ev_meta2.iterrows():
        _tk2 = _mrow['ticker']
        if _tk2 in _trade_cache:
            continue
        _csv2 = os.path.join(DATA_DIR, f'{_tk2.lower()}_trades.csv')
        if not os.path.exists(_csv2):
            continue
        try:
            _dfc = pd.read_csv(_csv2)
            if _dfc.empty or 'created_time' not in _dfc.columns:
                continue
            _dfc['created_time']      = pd.to_datetime(_dfc['created_time'], utc=True)
            _dfc['yes_price_dollars'] = pd.to_numeric(_dfc['yes_price_dollars'], errors='coerce')
            _dfc = _dfc.dropna(subset=['created_time','yes_price_dollars']).sort_values('created_time')
            if len(_dfc) > 0:
                _trade_cache[_tk2] = _dfc
        except Exception:
            pass
print(f'Cached {len(_trade_cache):,} trade frames.')
print()


def _band_at_buffer(buffer_h):
    'Compute above-floor longshot band stats at given buffer_h. Side-analysis only.'
    recs = []
    for _ev2, _ev_row2 in snap_ok.iterrows():
        _t_res2 = _ev_row2['t_resolve']
        if _t_res2 is None:
            continue
        _snap2   = pd.Timestamp(_t_res2) - pd.Timedelta(hours=buffer_h)
        _dom2    = _ev_row2['domain']
        _ev_meta3 = meta[meta['event_ticker'] == _ev2]
        for _, _mr in _ev_meta3.iterrows():
            _tk3 = _mr['ticker']
            _res3 = str(_mr.get('result', '')).strip()
            _st3  = str(_mr.get('status', '')).strip()
            if _st3 != 'finalized' or _res3 not in ('yes', 'no'):
                continue
            _df3 = _trade_cache.get(_tk3)
            if _df3 is None:
                continue
            _prior3 = _df3[_df3['created_time'] < _snap2]
            if _prior3.empty:
                continue
            _px3 = float(_prior3.iloc[-1]['yes_price_dollars'])
            recs.append({'event': _ev2, 'domain': _dom2,
                         'won': 1 if _res3 == 'yes' else 0,
                         'snapshot_price': _px3})
    _df_b = pd.DataFrame(recs)
    if _df_b.empty:
        return {'buffer_h': buffer_h, 'n_cand': 0, 'n_ev': 0, 'n_wins': 0,
                'win_rate': float('nan'), 'mean_implied': float('nan'),
                'gap': float('nan'), 'ci_lo': float('nan'), 'ci_hi': float('nan')}
    _ab_b   = _df_b[(_df_b['snapshot_price'] > FLOOR_MAX) & (_df_b['snapshot_price'] <= 0.15)]
    _n_b    = len(_ab_b)
    _nev_b  = _ab_b['event'].nunique()
    _nwin_b = int(_ab_b['won'].sum())
    _wr_b   = _ab_b['won'].mean()            if _n_b > 0 else float('nan')
    _mp_b   = _ab_b['snapshot_price'].mean() if _n_b > 0 else float('nan')
    if _n_b > 0 and _nwin_b > 0:
        _eg_b = {_e: _g for _e, _g in _ab_b.groupby('event')}
        _evs_b = list(_eg_b.keys())
        _boot_b = []
        for _ in range(BOOT_REPS):
            _s = np.random.choice(_evs_b, size=len(_evs_b), replace=True)
            _boot_b.append(float(pd.concat([_eg_b[_e] for _e in _s], ignore_index=True)['won'].mean()))
        _ci_lo_b, _ci_hi_b = np.percentile(_boot_b, [2.5, 97.5])
    elif _n_b > 0:
        _ci_lo_b = 0.0
        _ci_hi_b = clopper_pearson_upper(_n_b)
    else:
        _ci_lo_b = _ci_hi_b = float('nan')
    return {'buffer_h': buffer_h, 'n_cand': _n_b, 'n_ev': _nev_b, 'n_wins': _nwin_b,
            'win_rate': _wr_b, 'mean_implied': _mp_b,
            'gap': _wr_b - _mp_b if not (np.isnan(_wr_b) or np.isnan(_mp_b)) else float('nan'),
            'ci_lo': _ci_lo_b, 'ci_hi': _ci_hi_b}


_buffers = [24, 168, 336]          # 24h, 7d, 14d
_buf_results = []
for _bh in _buffers:
    print(f'  Running buffer = {_bh}h ({_bh//24}d)...', end=' ', flush=True)
    _buf_results.append(_band_at_buffer(_bh))
    print('done')
print()

print(f'  {"Buffer":>10}  {"n_cand":>7}  {"n_ev":>5}  {"win_rate":>9}  {"implied":>8}  {"gap":>7}  {"clustered 95%CI":^22}')
print('  ' + '-' * 82)
for _br in _buf_results:
    _bh_label = f'{_br["buffer_h"]}h ({_br["buffer_h"]//24}d)'
    _ci_str   = (f'[{_br["ci_lo"]:.4f}, {_br["ci_hi"]:.4f}]'
                 if not np.isnan(_br['ci_lo']) else '      (-)      ')
    _primary  = ' << PRIMARY' if _br['buffer_h'] == SNAPSHOT_BUFFER_H else ''
    print(f'  {_bh_label:>10}  {_br["n_cand"]:>7,}  {_br["n_ev"]:>5}  '
          f'{_br["win_rate"]:>9.4f}  {_br["mean_implied"]:>8.4f}  '
          f'{_br["gap"]:>+7.4f}  {_ci_str:^22}{_primary}')
print()
print('  Interpretation: if gap and CI direction are stable across buffers, the FLB')
print('  finding is robust to the snapshot timing assumption.')


CHECK 1 — Above-floor longshot band (2-15c): CI method verification

Above-floor longshot band  (0.02 < price ≤ 0.15)
  n_candidates      : 463
  n_events          : 135
  n_winners         : 14
  mean implied price: 0.0689
  win_rate (CW)     : 0.0302

  Naive Wilson 95%CI (treats 463 candidates as independent):
    [0.0181, 0.0501]
  Event-clustered block bootstrap 95%CI (135 events, 2000 reps):
    [0.0163, 0.0465]  << HEADLINE (already reported above)

  Clustered CI is narrower than Wilson (width: 0.0301 vs 0.0320).
  Both CIs lie entirely below mean implied (0.0689): YES

Events with a winner in the 2-15c band (14 winners):
  Event                                   Domain      Winner ticker                        Snap price
  --------------------------------------------------------------------------------------------------
  KXACAHOUSEVOTE-27JAN01                  Political   KXACAHOUSEVOTE-27JAN01-232               0.1000
  KXAUBCOACH-26                           Sports      KXA

Cached 2,105 trade frames.

  Running buffer = 24h (1d)... 

done
  Running buffer = 168h (7d)... 

done
  Running buffer = 336h (14d)... 

done

      Buffer   n_cand   n_ev   win_rate   implied      gap     clustered 95%CI    
  ----------------------------------------------------------------------------------
    24h (1d)      463    135     0.0302    0.0689  -0.0386     [0.0155, 0.0458]    << PRIMARY
   168h (7d)      443    109     0.0406    0.0720  -0.0314     [0.0263, 0.0568]   
  336h (14d)      396     95     0.0480    0.0722  -0.0242     [0.0308, 0.0655]   

  Interpretation: if gap and CI direction are stable across buffers, the FLB
  finding is robust to the snapshot timing assumption.


In [12]:
# ═══════════════════════════════════════════════════════════════════════════
# DISCRETE-ENTITY REFINEMENT
# Excludes the 24 numeric-range events flagged in Check 3 (value/seat/margin
# brackets). Primary result uses the clean discrete-entity set only.
# ═══════════════════════════════════════════════════════════════════════════

np.random.seed(RNG_SEED)

# 1. Build clean and numeric-range sets
_range_tickers  = [r[0] for r in _range_events]   # 24 from Check 3
cand_discrete   = cand[~cand['event'].isin(_range_tickers)].copy()
cand_range_only = cand[ cand['event'].isin(_range_tickers)].copy()

print('=' * 72)
print('1. DISCRETE-ENTITY SET')
print('=' * 72)
print()
print('Excluded numeric-range events (brackets, not distinct candidates):')
for _rt in sorted(_range_tickers):
    print(f'  {_rt}')
print()
_full_ev  = cand['event'].nunique()
_disc_ev  = cand_discrete['event'].nunique()
_range_ev = cand_range_only['event'].nunique()
print(f'  Full 122-event set   : {_full_ev} events   {len(cand):,} candidates')
print(f'  Excluded (numeric)   : {_range_ev} events   {len(cand_range_only):,} candidates')
print(f'  Discrete-entity set  : {_disc_ev} events   {len(cand_discrete):,} candidates')
print()
print('Discrete-entity set by domain:')
for _d2 in sorted(cand_discrete['domain'].unique()):
    _de = cand_discrete[cand_discrete['domain'] == _d2]['event'].nunique()
    _dc = len(cand_discrete[cand_discrete['domain'] == _d2])
    print(f'  {_d2:<12}: {_de:>3} events   {_dc:,} candidates')
print()
# 2. Primary above-floor band on discrete set
print('=' * 72)
print('2. ABOVE-FLOOR LONGSHOT BAND (2-15c) — DISCRETE-ENTITY PRIMARY RESULT')
print('=' * 72)
print()
_disc_ab = cand_discrete[
    (cand_discrete['snapshot_price'] > FLOOR_MAX) &
    (cand_discrete['snapshot_price'] <= 0.15)
].copy()
_dab = _band_stats(_disc_ab,
    'Above-floor (0.02 < price ≤ 0.15)  DISCRETE PRIMARY')
print()
# Explicit CI-vs-implied check
_clo = _dab['cw_lo']; _chi = _dab['cw_hi']; _mp = _dab['mean_px']
if not (np.isnan(_chi) or np.isnan(_mp)):
    if _chi < _mp:
        print(f'  EXPLICIT CHECK: clustered CI [{_clo:.4f}, {_chi:.4f}] '
              f'lies ENTIRELY BELOW implied {_mp:.4f}. FLB confirmed on discrete set.')
    else:
        print(f'  EXPLICIT CHECK: clustered CI [{_clo:.4f}, {_chi:.4f}] '
              f'does NOT lie entirely below implied {_mp:.4f}.')
print()
# 3. Domain split on discrete set
print('=' * 72)
print('3. ABOVE-FLOOR BAND BY DOMAIN — DISCRETE SET')
print('=' * 72)
print()
for _dom3 in ('Sports', 'Political'):
    _dom_sub = cand_discrete[cand_discrete['domain'] == _dom3]
    _dom_ab  = _dom_sub[
        (_dom_sub['snapshot_price'] > FLOOR_MAX) &
        (_dom_sub['snapshot_price'] <= 0.15)
    ].copy()
    if len(_dom_ab) > 0:
        _band_stats(_dom_ab, f'Above-floor (2-15c)  {_dom3} only  (discrete set)')
    else:
        print(f'  {_dom3}: no above-floor candidates after exclusion.')
    print()
# 4. Numeric-range events as their own pool (secondary / for paper)
print('=' * 72)
print('4. NUMERIC-RANGE POOL — SECONDARY (24 events, NOT pooled with primary)')
print('=' * 72)
print('Sub-markets are value/seat/margin brackets, not discrete candidates.')
print()
_range_ab = cand_range_only[
    (cand_range_only['snapshot_price'] > FLOOR_MAX) &
    (cand_range_only['snapshot_price'] <= 0.15)
].copy()
if len(_range_ab) > 0:
    _rab = _band_stats(_range_ab,
        'Above-floor (0.02 < price ≤ 0.15)  NUMERIC-RANGE POOL')
else:
    print('  No above-floor candidates in the numeric-range pool.')
    _rab = {'wr_cw': float('nan'), 'cw_lo': float('nan'),
             'cw_hi': float('nan'), 'mean_px': float('nan')}
print()
print('=' * 72)
print('SUMMARY COMPARISON  (above-floor 2-15c band)')
print('=' * 72)
print(f'  {"Pool":<32}  {"N_cand":>7}  {"N_ev":>5}  '
      f'{"WinRate":>8}  {"Implied":>8}  {"Gap":>7}  Clustered 95%CI')
print('  ' + '-' * 90)
# Full 122-event set (from _band_results)
_ab_full = next(r for r in _band_results if 'Above-floor' in r['band'])
_rows_cmp = [
    ('Full 122-event set',    _ab_full['n'],      _ab_full['n_ev'],
     _ab_full['wr_cw'], _ab_full['mean_px'], _ab_full['cw_lo'], _ab_full['cw_hi']),
    ('Discrete-entity (' + str(_disc_ev) + ' ev)',
     _dab['n'],   _dab['n_ev'],  _dab['wr_cw'],  _dab['mean_px'],  _dab['cw_lo'],  _dab['cw_hi']),
    ('Numeric-range (' + str(_range_ev) + ' ev)',
     _rab.get('n', 0), _rab.get('n_ev', 0),
     _rab.get('wr_cw', float('nan')), _rab.get('mean_px', float('nan')),
     _rab.get('cw_lo', float('nan')), _rab.get('cw_hi', float('nan'))),
]
for _lbl, _n, _ne, _wr, _mp2, _lo, _hi in _rows_cmp:
    _ci = f'[{_lo:.4f}, {_hi:.4f}]' if not np.isnan(_lo) else '     (-)     '
    _gp = _wr - _mp2 if not (np.isnan(_wr) or np.isnan(_mp2)) else float('nan')
    _gp_s = f'{_gp:+.4f}' if not np.isnan(_gp) else '      -'
    print(f'  {_lbl:<32}  {_n:>7,}  {_ne:>5}  '
          f'{_wr:>8.4f}  {_mp2:>8.4f}  {_gp_s:>7}  {_ci}')
print()
# Save discrete outputs (suffix _discrete, does not overwrite _full)
cand_discrete.to_csv(os.path.join(DATA_DIR, 'candidate_level_discrete.csv'), index=False)
print(f'Saved {len(cand_discrete):,} rows -> data/candidate_level_discrete.csv')
print()
print('Running _make_tables on discrete set and saving bucket CSV...')
_tbl_cw_d, _ = _make_tables(cand_discrete)
_fp_d = os.path.join(OUT_DIR, 'buckets_overall_cw_discrete.csv')
_tbl_cw_d.to_csv(_fp_d, index=False)
print('Saved buckets_overall_cw_discrete.csv -> output/')


1. DISCRETE-ENTITY SET

Excluded numeric-range events (brackets, not distinct candidates):
  KXACAHOUSEVOTE-27JAN01
  KXCDASEATS-25OCT29
  KXCHILEANRUNOFF-25DEC14
  KXELECTIONMOVABQ-25DEC09
  KXELECTIONMOVAZ7S-25SEP23
  KXELECTIONMOVIRELAND-25OCT24
  KXELECTIONMOVMIAMI-25DEC09
  KXGORTONDENTONMOV-26FEB26
  KXGOVTFUNDSVOTES-27JAN01
  KXGROENSEATS-25OCT29
  KXHONDURASPRESIDENTMOV-25NOV30
  KXJERSEYCITMOV-25DEC02
  KXLDPSEATS-26FEB08
  KXLOUDOUNMOVVAGOV-25NOV04
  KXMAINEHOUSE94SPECIAL-26FEB24
  KXMOLDOVABEP-25SEP28
  KXMOLDOVAPAS-25SEP28
  KXMOVCOSTARICAPRESR1-26FEB01
  KXMOVNJ11SPECIALD-26FEB05
  KXMOVPARISRUNOFF-26MAR22
  KXMOVPORTUGALPRESRUNOFF-26FEB08
  KXMOVTX18SPECIAL-26JAN31
  KXNJASSEMBLYSEATSD-25NOV04
  KXNORWAYCON-25SEP08
  KXNORWAYLABOUR-25SEP08
  KXNOTSEEKREELECTION-26MAR01
  KXNYCMAYOR-25NOV04
  KXPARIS1RMOV-26MAR15
  KXPVVSEATS-25OCT29
  KXSEATTLEMAYORMOV-25NOV04
  KXTN7SMOV-25NOV07
  KXTXSENDPRIMARYMOV-26MAR03
  KXTXSENRPRIMARYMOV-26MAR03
  KXVAAGMOV-25NOV04
  KXVAHOD1-26NO

  Above-floor (0.02 < price ≤ 0.15)  DISCRETE PRIMARY
    n_candidates : 369   n_events : 104   share_at_floor : 0.000
    mean_implied : 0.0664
    win_rate CW  : 0.0271  95%CI [0.0122, 0.0446]  gap vs implied: -0.0393  (bootstrap)
    win_rate EW  : 0.0317  95%CI [0.0130, 0.0471]  gap vs implied: -0.0347  (bootstrap)


  EXPLICIT CHECK: clustered CI [0.0122, 0.0446] lies ENTIRELY BELOW implied 0.0664. FLB confirmed on discrete set.

3. ABOVE-FLOOR BAND BY DOMAIN — DISCRETE SET



  Above-floor (2-15c)  Sports only  (discrete set)
    n_candidates : 208   n_events : 34   share_at_floor : 0.000
    mean_implied : 0.0609
    win_rate CW  : 0.0192  95%CI [0.0047, 0.0383]  gap vs implied: -0.0416  (bootstrap)
    win_rate EW  : 0.0225  95%CI [0.0034, 0.0383]  gap vs implied: -0.0383  (bootstrap)




  Above-floor (2-15c)  Political only  (discrete set)
    n_candidates : 161   n_events : 70   share_at_floor : 0.000
    mean_implied : 0.0736
    win_rate CW  : 0.0373  95%CI [0.0120, 0.0671]  gap vs implied: -0.0363  (bootstrap)
    win_rate EW  : 0.0362  95%CI [0.0109, 0.0576]  gap vs implied: -0.0374  (bootstrap)


4. NUMERIC-RANGE POOL — SECONDARY (24 events, NOT pooled with primary)
Sub-markets are value/seat/margin brackets, not discrete candidates.



  Above-floor (0.02 < price ≤ 0.15)  NUMERIC-RANGE POOL
    n_candidates : 94   n_events : 31   share_at_floor : 0.000
    mean_implied : 0.0785
    win_rate CW  : 0.0426  95%CI [0.0095, 0.0909]  gap vs implied: -0.0360  (bootstrap)
    win_rate EW  : 0.0511  95%CI [0.0125, 0.0833]  gap vs implied: -0.0274  (bootstrap)


SUMMARY COMPARISON  (above-floor 2-15c band)
  Pool                               N_cand   N_ev   WinRate   Implied      Gap  Clustered 95%CI
  ------------------------------------------------------------------------------------------
  Full 122-event set                    463    135    0.0302    0.0689  -0.0386  [0.0163, 0.0465]
  Discrete-entity (133 ev)              369    104    0.0271    0.0664  -0.0393  [0.0122, 0.0446]
  Numeric-range (36 ev)                  94     31    0.0426    0.0785  -0.0360  [0.0095, 0.0909]

Saved 1,579 rows -> data/candidate_level_discrete.csv

Running _make_tables on discrete set and saving bucket CSV...


Saved buckets_overall_cw_discrete.csv -> output/
